This notebook is the code for:
1. Hand Detection
2. Active Object Detection
3. Generic Object Detection
4. Interactive Pair Generation
5. Finally the output should be an annotation dict as needed by the feature generator.

In [2]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import _init_paths
import os
import sys
import numpy as np
import argparse
import pprint
import pdb
import time
import cv2
import torch
from torch.autograd import Variable
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F 
from PIL import Image

import torchvision.transforms as transforms
import torchvision.datasets as dset
# from scipy.misc import imread
from roi_data_layer.roidb import combined_roidb
from roi_data_layer.roibatchLoader import roibatchLoader
from model.utils.config import cfg, cfg_from_file, cfg_from_list, get_output_dir
from model.rpn.bbox_transform import clip_boxes
# from model.nms.nms_wrapper import nms
from model.roi_layers import nms
from model.rpn.bbox_transform import bbox_transform_inv
from model.utils.net_utils import save_net, load_net, vis_detections, vis_detections_PIL, vis_detections_filtered_objects_PIL, vis_detections_filtered_objects # (1) here add a function to viz
from model.utils.blob import im_list_to_blob
from model.faster_rcnn.vgg16 import vgg16
from model.faster_rcnn.resnet import resnet
import pdb

# Assuming all necessary imports from the script are included above


# Add this function definition before you run the hand object detection function
def _get_image_blob(im):
    """Converts an image into a network input."""
    im_orig = im.astype(np.float32, copy=True)
    im_orig -= cfg.PIXEL_MEANS

    im_shape = im_orig.shape
    im_size_min = np.min(im_shape[0:2])
    im_size_max = np.max(im_shape[0:2])

    # Scale the image so that its smaller dimension equals the target size
    processed_ims = []
    im_scale_factors = []

    for target_size in cfg.TEST.SCALES:
        im_scale = float(target_size) / float(im_size_min)
        # Prevent the biggest axis from being more than MAX_SIZE
        if np.round(im_scale * im_size_max) > cfg.TEST.MAX_SIZE:
            im_scale = float(cfg.TEST.MAX_SIZE) / float(im_size_max)
        im = cv2.resize(im_orig, None, None, fx=im_scale, fy=im_scale,
                        interpolation=cv2.INTER_LINEAR)
        im_scale_factors.append(im_scale)
        processed_ims.append(im)

    # Create a blob to hold the input images
    blob = im_list_to_blob(processed_ims)

    return blob, np.array(im_scale_factors)


# Helper function for converting detections to human-readable format
def convert_detections_to_human_readable(obj_dets, hand_dets, thresh_hand=0.5, thresh_obj=0.5):
    human_readable_output = {
        "objects": [],
        "hands": []
    }

    def bbox_to_human_readable(bbox):
        return {
            "xmin": int(np.round(bbox[0])),
            "ymin": int(np.round(bbox[1])),
            "xmax": int(np.round(bbox[2])),
            "ymax": int(np.round(bbox[3]))
        }

    # Process object detections
    if obj_dets is not None:
        for i in range(np.minimum(10, obj_dets.shape[0])):
            score = obj_dets[i, 4]
            if score > thresh_obj:
                bbox = obj_dets[i, :4]
                contact_state = obj_dets[i, 5]
                offset_vector = obj_dets[i, 6:9]
                human_readable_output["objects"].append({
                    "bounding_box": bbox_to_human_readable(bbox),
                    "score": score,
                    "contact_state": contact_state,
                    "offset_vector": offset_vector.tolist()
                })

    # Process hand detections
    if hand_dets is not None:
        for i in range(np.minimum(10, hand_dets.shape[0])):
            score = hand_dets[i, 4]
            if score > thresh_hand:
                bbox = hand_dets[i, :4]
                contact_state = hand_dets[i, 5]
                offset_vector = hand_dets[i, 6:9]
                lr = "left" if hand_dets[i, -1] == 0 else "right"
                human_readable_output["hands"].append({
                    "bounding_box": bbox_to_human_readable(bbox),
                    "score": score,
                    "contact_state": "in contact" if contact_state > 0 else "not in contact",
                    "offset_vector": offset_vector.tolist(),
                    "hand_side": lr
                })

    return human_readable_output

def run_hand_object_detection(args):
    # Load model
    model_dir = args['load_dir'] + "/" + args['net'] + "_handobj_100K" + "/" + args['dataset']
    if not os.path.exists(model_dir):
        raise Exception('There is no input directory for loading network from ' + model_dir)
    load_name = os.path.join(model_dir, 'faster_rcnn_{}_{}_{}.pth'.format(args['checksession'], args['checkepoch'], args['checkpoint']))

    pascal_classes = np.asarray(['__background__', 'targetobject', 'hand']) 
    args['set_cfgs'] = ['ANCHOR_SCALES', '[8, 16, 32, 64]', 'ANCHOR_RATIOS', '[0.5, 1, 2]'] 

    # Initialize the network
    if args['net'] == 'vgg16':
        fasterRCNN = vgg16(pascal_classes, pretrained=False, class_agnostic=args['class_agnostic'])
    elif args['net'] == 'res101':
        fasterRCNN = resnet(pascal_classes, 101, pretrained=False, class_agnostic=args['class_agnostic'])
    elif args['net'] == 'res50':
        fasterRCNN = resnet(pascal_classes, 50, pretrained=False, class_agnostic=args['class_agnostic'])
    elif args['net'] == 'res152':
        fasterRCNN = resnet(pascal_classes, 152, pretrained=False, class_agnostic=args['class_agnostic'])
    else:
        raise ValueError("network is not defined")

    fasterRCNN.create_architecture()

    print("load checkpoint %s" % (load_name))
    checkpoint = torch.load(load_name, map_location=(lambda storage, loc: storage))
    fasterRCNN.load_state_dict(checkpoint['model'])
    if 'pooling_mode' in checkpoint.keys():
        cfg.POOLING_MODE = checkpoint['pooling_mode']

    print('load model successfully!')

    # Initialize tensor holders
    im_data = torch.FloatTensor(1)
    im_info = torch.FloatTensor(1)
    num_boxes = torch.LongTensor(1)
    gt_boxes = torch.FloatTensor(1)
    box_info = torch.FloatTensor(1) 

    # Ship to CUDA if necessary

    with torch.no_grad():

        fasterRCNN.eval()

        max_per_image = 100
        thresh_hand = args['thresh_hand']
        thresh_obj = args['thresh_obj']
        vis = args['vis']

        imglist = [args['image_path']]
        im_file = imglist[0]
        im_in = cv2.imread(im_file)
        im = im_in

        blobs, im_scales = _get_image_blob(im)
        im_blob = blobs
        im_info_np = np.array([[im_blob.shape[1], im_blob.shape[2], im_scales[0]]], dtype=np.float32)

        im_data_pt = torch.from_numpy(im_blob).permute(0, 3, 1, 2)
        im_info_pt = torch.from_numpy(im_info_np)

        im_data.resize_(im_data_pt.size()).copy_(im_data_pt)
        im_info.resize_(im_info_pt.size()).copy_(im_info_pt)
        gt_boxes.resize_(1, 1, 5).zero_()
        num_boxes.resize_(1).zero_()
        box_info.resize_(1, 1, 5).zero_() 

        # Forward pass through the network
        rois, cls_prob, bbox_pred, \
        rpn_loss_cls, rpn_loss_box, \
        RCNN_loss_cls, RCNN_loss_bbox, \
        rois_label, loss_list = fasterRCNN(im_data, im_info, gt_boxes, num_boxes, box_info) 

        scores = cls_prob.data
        boxes = rois.data[:, :, 1:5]

        # Extract predicted params
        contact_vector = loss_list[0][0] # hand contact state info
        offset_vector = loss_list[1][0].detach() # offset vector (factored into a unit vector and a magnitude)
        lr_vector = loss_list[2][0].detach() # hand side info (left/right)

        # Get hand contact
        _, contact_indices = torch.max(contact_vector, 2)
        contact_indices = contact_indices.squeeze(0).unsqueeze(-1).float()

        # Get hand side
        lr = torch.sigmoid(lr_vector) > 0.5
        lr = lr.squeeze(0).float()

        if cfg.TEST.BBOX_REG:
            # Apply bounding-box regression deltas
            box_deltas = bbox_pred.data
            if cfg.TRAIN.BBOX_NORMALIZE_TARGETS_PRECOMPUTED:
                # Optionally normalize targets by a precomputed mean and stdev
                if args['class_agnostic']:
                    box_deltas = box_deltas.view(-1, 4) * torch.FloatTensor(cfg.TRAIN.BBOX_NORMALIZE_STDS) \
                              + torch.FloatTensor(cfg.TRAIN.BBOX_NORMALIZE_MEANS)
                    box_deltas = box_deltas.view(1, -1, 4)
                else:
                    box_deltas = box_deltas.view(-1, 4) * torch.FloatTensor(cfg.TRAIN.BBOX_NORMALIZE_STDS) \
                              + torch.FloatTensor(cfg.TRAIN.BBOX_NORMALIZE_MEANS)
                    box_deltas = box_deltas.view(1, -1, 4 * len(pascal_classes))
            pred_boxes = bbox_transform_inv(boxes, box_deltas, 1)
            pred_boxes = clip_boxes(pred_boxes, im_info.data, 1)
        else:
            # Simply repeat the boxes, once for each class
            pred_boxes = np.tile(boxes, (1, scores.shape[1]))

        pred_boxes /= im_scales[0]

        scores = scores.squeeze()
        pred_boxes = pred_boxes.squeeze()

        obj_dets, hand_dets = None, None
        for j in range(1, len(pascal_classes)):
            if pascal_classes[j] == 'hand':
                inds = torch.nonzero(scores[:,j]>thresh_hand).view(-1)
            elif pascal_classes[j] == 'targetobject':
                inds = torch.nonzero(scores[:,j]>thresh_obj).view(-1)

            if inds.numel() > 0:
                cls_scores = scores[:,j][inds]
                _, order = torch.sort(cls_scores, 0, True)
                if args['class_agnostic']:
                    cls_boxes = pred_boxes[inds, :]
                else:
                    cls_boxes = pred_boxes[inds][:, j * 4:(j + 1) * 4]

                cls_dets = torch.cat((cls_boxes, cls_scores.unsqueeze(1), contact_indices[inds], offset_vector.squeeze(0)[inds], lr[inds]), 1)
                cls_dets = cls_dets[order]
                keep = nms(cls_boxes[order, :], cls_scores[order], cfg.TEST.NMS)
                cls_dets = cls_dets[keep.view(-1).long()]
                if pascal_classes[j] == 'targetobject':
                    obj_dets = cls_dets.cpu().numpy()
                if pascal_classes[j] == 'hand':
                    hand_dets = cls_dets.cpu().numpy()

        human_readable_output = convert_detections_to_human_readable(obj_dets, hand_dets)
        return human_readable_output  # Return instead of printing

# Function to run the model in a Jupyter notebook cell
def detect_objects_and_hands_in_image(args_dict):
    output = run_hand_object_detection(args_dict)
    return output




import json
import torch
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Assuming detect_objects_and_hands_in_image and other necessary imports are already defined

def run_hand_object_detector(image_path, checkpoint, checkepoch):
    # Prepare the arguments for the hand-object detector function
    args_dict = {
        'dataset': 'pascal_voc',
        'cfg_file': 'cfgs/res101.yml',
        'net': 'res101',
        'set_cfgs': None,
        'load_dir': "models",
        'image_path': image_path,
        'save_dir': "images_det",
        'cuda': False,
        'mGPUs': False,
        'class_agnostic': False,
        'parallel_type': 0,
        'checksession': 1,
        'checkepoch': checkepoch,
        'checkpoint': checkpoint,
        'batch_size': 1,
        'vis': True,
        'webcam_num': -1,
        'thresh_hand': 0.5,
        'thresh_obj': 0.5
    }
    
    # Call the hand-object detector function
    result = detect_objects_and_hands_in_image(args_dict)
    return result

def detect_objects_from_image_path(image_path):
    # Load the pre-trained Faster R-CNN model
    model = models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    model.eval()
    
    # Define the image transformation
    transform = transforms.Compose([
        transforms.ToTensor()
    ])
    
    # Load the image from the file path
    img = Image.open(image_path).convert("RGB")
    
    # Transform the image
    img_tensor = transform(img)
    
    # Add a batch dimension
    img_tensor = img_tensor.unsqueeze(0)
    
    # Perform object detection
    with torch.no_grad():
        predictions = model(img_tensor)
    
    # Extract the predicted bounding boxes and labels
    boxes = predictions[0]['boxes']
    scores = predictions[0]['scores']
    
    # Set a confidence threshold (e.g., 0.5)
    threshold = 0.5
    selected_boxes = boxes[scores > threshold]
    selected_scores = scores[scores > threshold]
    
    return selected_boxes, selected_scores, img


import numpy as np

def compute_box_center(box):
    """Compute the center of a bounding box."""
    x1, y1, x2, y2 = box
    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2
    return np.array([center_x, center_y])

def filter_closest_boxes(generic_boxes, hand_boxes, max_boxes=4):
    """Filter generic boxes to keep only the closest ones to the hand boxes."""
    closest_boxes = []
    
    for generic_box in generic_boxes:
        generic_center = compute_box_center(generic_box)
        
        # Compute the distance to each hand box and keep the minimum distance
        min_distance = float('inf')
        for hand_box in hand_boxes:
            hand_center = compute_box_center(hand_box)
            distance = np.linalg.norm(generic_center - hand_center)
            if distance < min_distance:
                min_distance = distance
        
        closest_boxes.append((min_distance, generic_box))
    
    # Sort by the minimum distance and take the closest max_boxes
    closest_boxes.sort(key=lambda x: x[0])
    filtered_boxes = [box for _, box in closest_boxes[:max_boxes]]
    
    return filtered_boxes

def combine_detections(image_path, checkpoint, checkepoch):
    # Run hand-object detector
    hand_object_output = run_hand_object_detector(image_path, checkpoint, checkepoch)
    
    # Run generic object detector
    generic_boxes, generic_scores, img = detect_objects_from_image_path(image_path)
    
    # Extract hand boxes
    hand_boxes = [np.array([hand['bounding_box']['xmin'], hand['bounding_box']['ymin'],
                            hand['bounding_box']['xmax'], hand['bounding_box']['ymax']])
                  for hand in hand_object_output['hands']]
    
    # Filter generic boxes to keep only the closest max 4 to the hand boxes
    filtered_generic_boxes = filter_closest_boxes(generic_boxes, hand_boxes, max_boxes=4)
    
    # Initialize the result dictionary
    result_dict = {'bboxes': {}}
    box_id = 0
    
    # Add hand-object detector results to the dictionary
    for hand in hand_object_output['hands']:
        x1, y1 = hand['bounding_box']['xmin'], hand['bounding_box']['ymin']
        x2, y2 = hand['bounding_box']['xmax'], hand['bounding_box']['ymax']
        category = f"{hand['hand_side']}_hand"
        result_dict['bboxes'][str(box_id)] = {'class': 'hand', 'category': category, 'bbox': [x1, y1, x2, y2]}
        box_id += 1
    
    for obj in hand_object_output['objects']:
        x1, y1 = obj['bounding_box']['xmin'], obj['bounding_box']['ymin']
        x2, y2 = obj['bounding_box']['xmax'], obj['bounding_box']['ymax']
        result_dict['bboxes'][str(box_id)] = {'class': 'active_object', 'category': 'None', 'bbox': [x1, y1, x2, y2]}
        box_id += 1
    
    # Add filtered generic object detector results to the dictionary
    for box in filtered_generic_boxes:
        x1, y1, x2, y2 = box
        result_dict['bboxes'][str(box_id)] = {'class': 'generic_object', 'category': 'None', 'bbox': [x1, y1, x2, y2]}
        box_id += 1
    
    # Return the result dictionary
    return result_dict


import numpy as np

def compute_box_center(box):
    """Compute the center of a bounding box."""
    x1, y1, x2, y2 = box
    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2
    return np.array([center_x, center_y])

def compute_distance(box1, box2):
    """Compute the Euclidean distance between the centers of two bounding boxes."""
    center1 = compute_box_center(box1)
    center2 = compute_box_center(box2)
    return np.linalg.norm(center1 - center2)

def find_closest_pair(entity1_list, entity2_list):
    """Find the closest pair between two lists of entities based on their bounding boxes."""
    min_distance = float('inf')
    best_pair = (None, None)
    
    for id1, entity1 in entity1_list.items():
        for id2, entity2 in entity2_list.items():
            dist = compute_distance(entity1['bbox'], entity2['bbox'])
            if dist < min_distance:
                min_distance = dist
                best_pair = (id1, id2)
    
    return best_pair

def identify_interactive_pairs(detection_dict):
    hands = {k: v for k, v in detection_dict['bboxes'].items() if v['class'] == 'hand'}
    active_objects = {k: v for k, v in detection_dict['bboxes'].items() if v['class'] == 'active_object'}
    generic_objects = {k: v for k, v in detection_dict['bboxes'].items() if v['class'] == 'generic_object'}
    
    pairs = []
    
    # Step 1: Pair hands with the closest active objects
    while hands:
        hand_id, active_obj_id = find_closest_pair(hands, active_objects)
        if hand_id is not None and active_obj_id is not None:
            pairs.append([hand_id, active_obj_id])
            del hands[hand_id]
    
    # Step 2: Pair remaining hands with the closest generic objects
    while hands and generic_objects:
        hand_id, generic_obj_id = find_closest_pair(hands, generic_objects)
        if hand_id is not None and generic_obj_id is not None:
            pairs.append([hand_id, generic_obj_id])
            del hands[hand_id]
            del generic_objects[generic_obj_id]
    
    # Step 3: Pair all active objects with the closest generic objects
    while active_objects and generic_objects:
        active_obj_id, generic_obj_id = find_closest_pair(active_objects, generic_objects)
        if active_obj_id is not None and generic_obj_id is not None:
            pairs.append([active_obj_id, generic_obj_id])
            del active_objects[active_obj_id]
            del generic_objects[generic_obj_id]
    
    return pairs


def generate_annotation_file(image_path):
    # Example usage
    
    checkpoint = 89999
    checkepoch = 8

    object_detection_dict = combine_detections(image_path, checkpoint, checkepoch)
    interactive_pairs = identify_interactive_pairs(object_detection_dict)
    
    metadata = {}
    metadata['image_path'] = image_path
    
    annotation_data = {}
    annotation_data['bboxes'] = object_detection_dict
    annotation_data['metadata'] = metadata
    annotation_data['interactive_pairs'] = interactive_pairs

    return annotation_data

{'metadata': {'activity name': 'FuelCar',
  'yt_id': 'yk4yE_TawLM',
  'frame no.': '3744'},
 'bboxes': {'0': {'class': 'generic_object', 'bbox': [246, 205, 300, 288]},
  '1': {'class': 'generic_object', 'bbox': [287, 262, 325, 293]},
  '2': {'class': 'generic_object', 'bbox': [262, 44, 475, 351]},
  '3': {'class': 'hand', 'bbox': [217, 218, 249, 257]}},
 'relations': [[[0, 2],
   {'scr': ['Contact'],
    'lr': ['Behind/Front'],
    'mr': ['Negligible Relative Motion']}],
  [[3, 0],
   {'scr': ['No Contact'], 'lr': ['Behind/Front'], 'mr': ['Moving Toward']}],
  [[3, 2],
   {'scr': ['No Contact'],
    'lr': ['Behind/Front', 'Left/Right'],
    'mr': ['Moving Toward']}],
  [[0, 1],
   {'scr': ['Contact'],
    'lr': ['Left/Right'],
    'mr': ['Negligible Relative Motion']}],
  [[1, 2],
   {'scr': ['Contact'],
    'lr': ['Behind/Front', 'Inside'],
    'mr': ['Negligible Relative Motion']}],
  [[3, 1],
   {'scr': ['No Contact'],
    'lr': ['Below/Above', 'Left/Right'],
    'mr': ['Moving Toward']}]]}

In [3]:
# Example usage
image_path = '/Users/aunmesh/Desktop/code_chi/hand_object_detector/images/IMG_4228.png'

annotation_data = generate_annotation_file(image_path)

load checkpoint models/res101_handobj_100K/pascal_voc/faster_rcnn_1_8_89999.pth
load model successfully!


/opt/anaconda3/envs/handobj_new/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/handobj_new/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
annotation_data

{'bboxes': {'bboxes': {'0': {'class': 'hand',
    'category': 'right_hand',
    'bbox': [879, 2635, 1467, 3029]},
   '1': {'class': 'hand',
    'category': 'left_hand',
    'bbox': [1691, 2478, 2335, 2969]},
   '2': {'class': 'active_object',
    'category': 'None',
    'bbox': [749, 2500, 2568, 3763]},
   '3': {'class': 'generic_object',
    'category': 'None',
    'bbox': [tensor(946.8057),
     tensor(1327.6459),
     tensor(3024.),
     tensor(3936.5593)]},
   '4': {'class': 'generic_object',
    'category': 'None',
    'bbox': [tensor(909.4440),
     tensor(2329.4180),
     tensor(2738.9932),
     tensor(3696.4949)]},
   '5': {'class': 'generic_object',
    'category': 'None',
    'bbox': [tensor(1144.1019),
     tensor(2778.6467),
     tensor(2277.9968),
     tensor(3592.7180)]},
   '6': {'class': 'generic_object',
    'category': 'None',
    'bbox': [tensor(27.3313),
     tensor(590.9559),
     tensor(2516.1372),
     tensor(3858.3347)]}}},
 'metadata': {'image_path': '/Users/au